In [1]:
# ============================================================
# DUAL-MODE TACTICAL PRODUCTION SYSTEM
# Momentum Bucket  → Original (no filter)
# Quality Bucket   → Mild Trend Filter (Close > 50-SMA)
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ====================== SETTINGS ======================
Z_ENTRY     = -1.5
Z_EXIT      = 0.0
ATR_MULT    = 2.0
SMA_WINDOW  = 20
ATR_WINDOW  = 14
TREND_SMA   = 50

# ===== BUCKETS =====
MOMENTUM_BUCKET = ["NVDA", "META", "NET"]
QUALITY_BUCKET  = ["NOW", "MSFT", "GOOGL", "PANW", "CRWD", "DDOG", "CRM"]
# ======================================================


def get_data(ticker, days=150):
    end = datetime.now().date() + timedelta(days=1)
    start = end - timedelta(days=days)
    try:
        df = yf.download(ticker, start=start.strftime('%Y-%m-%d'),
                         end=end.strftime('%Y-%m-%d'),
                         multi_level_index=False, progress=False, auto_adjust=True)
        if df.empty:
            return None
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna().copy()
        df.index = pd.to_datetime(df.index)
        return df
    except Exception as e:
        print(f"Error {ticker}: {e}")
        return None


def add_indicators(df):
    df = df.copy()
    df['sma'] = df['Close'].rolling(SMA_WINDOW).mean()
    df['std'] = df['Close'].rolling(SMA_WINDOW).std()
    df['zscore'] = (df['Close'] - df['sma']) / df['std']
    df['trend_sma'] = df['Close'].rolling(TREND_SMA).mean()

    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - df['Close'].shift(1)).abs(),
        (df['Low'] - df['Close'].shift(1)).abs()
    ], axis=1).max(axis=1)
    df['atr'] = tr.rolling(ATR_WINDOW).mean()
    return df


def get_levels(ticker, use_trend_filter=False):
    df = get_data(ticker)
    if df is None or len(df) < 60:
        return None

    df = add_indicators(df)
    last = df.iloc[-1]

    close = float(last['Close'])
    sma   = float(last['sma'])
    std   = float(last['std'])
    atr   = float(last['atr'])
    z     = float(last['zscore'])
    trend_sma = float(last['trend_sma']) if pd.notna(last['trend_sma']) else None

    buy_trigger   = sma + (Z_ENTRY * std)
    initial_stop  = buy_trigger - (ATR_MULT * atr)
    mean_exit     = sma
    current_stop  = close - (ATR_MULT * atr)

    trend_ok = True
    if use_trend_filter and trend_sma is not None:
        trend_ok = close > trend_sma

    risk   = buy_trigger - initial_stop
    reward = mean_exit - buy_trigger
    rr     = reward / risk if risk > 0 else 0

    return {
        'ticker': ticker,
        'close': close,
        'zscore': z,
        'sma': sma,
        'atr': atr,
        'trend_sma': trend_sma,
        'buy_trigger': buy_trigger,
        'initial_stop': initial_stop,
        'mean_exit': mean_exit,
        'current_stop': current_stop,
        'trend_ok': trend_ok,
        'use_filter': use_trend_filter,
        'risk': risk,
        'reward': reward,
        'rr': rr,
        'signal': (z < Z_ENTRY) and trend_ok
    }


def print_stock_levels(info):
    if info is None:
        print("  → Data unavailable\n")
        return

    mode = "QUALITY (50-SMA Filter)" if info['use_filter'] else "MOMENTUM (No Filter)"
    signal_str = "*** ALL-IN BUY SIGNAL ***" if info['signal'] else "No signal yet"

    print(f"\n{'='*72}")
    print(f"  {info['ticker']}   |   {mode}")
    print(f"{'='*72}")
    print(f"  Current Price      : ${info['close']:.2f}")
    print(f"  Z-score            : {info['zscore']:.2f}   (entry when < {Z_ENTRY})")
    print(f"  20-SMA             : ${info['sma']:.2f}")
    if info['trend_sma']:
        print(f"  50-SMA             : ${info['trend_sma']:.2f}   (Trend OK: {info['trend_ok']})")
    print(f"  ATR(14)            : ${info['atr']:.2f}")
    print(f"{'-'*72}")
    print(f"  BUY TRIGGER        : ${info['buy_trigger']:.2f}")
    print(f"     → This is where Z-score = {Z_ENTRY}")
    print()
    print(f"  IF YOU BUY AT THE TRIGGER:")
    print(f"     Entry Price         : ${info['buy_trigger']:.2f}")
    print(f"     Initial Stop Loss   : ${info['initial_stop']:.2f}")
    print(f"     Mean-Reversion Exit : ${info['mean_exit']:.2f}  (Z > 0)")
    print(f"     Risk / Reward       : ${info['risk']:.2f} / ${info['reward']:.2f}  →  {info['rr']:.2f}:1")
    print(f"{'-'*72}")
    print(f"  Status: {signal_str}")
    if info['signal']:
        print(f"  → GO ALL-IN")
        print(f"  → Place initial stop around ${info['current_stop']:.2f}")
    else:
        dist = info['close'] - info['buy_trigger']
        pct  = dist / info['close'] * 100
        print(f"  → Needs to fall ${dist:.2f} ({pct:.1f}%) to trigger")
    print(f"{'='*72}\n")


def live_status_all():
    """Run this every day"""
    print("\n" + "#"*80)
    print(f"DUAL-MODE LIVE DASHBOARD   |   {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print("#"*80)

    print("\n" + "█"*80)
    print("  MOMENTUM BUCKET  →  Original Strategy (No Filter)")
    print("  " + ", ".join(MOMENTUM_BUCKET))
    print("█"*80)

    for t in MOMENTUM_BUCKET:
        info = get_levels(t, use_trend_filter=False)
        print_stock_levels(info)

    print("\n" + "█"*80)
    print("  QUALITY BUCKET  →  Mild Trend Filter (Close > 50-SMA)")
    print("  " + ", ".join(QUALITY_BUCKET))
    print("█"*80)

    for t in QUALITY_BUCKET:
        info = get_levels(t, use_trend_filter=True)
        print_stock_levels(info)

    print("\n" + "#"*80)
    print("RULES SUMMARY")
    print("#"*80)
    print("MOMENTUM (NVDA, META, NET):")
    print("   Entry : Z < -1.5")
    print("   Stop  : Trail = Close – 2×ATR (only raise)")
    print("   Exit  : Trailing stop hit  OR  Z > 0")
    print()
    print("QUALITY (NOW, MSFT, GOOGL, PANW, CRWD, DDOG, CRM):")
    print("   Entry : Z < -1.5  AND  Close > 50-SMA")
    print("   Stop  : Trail = Close – 2×ATR (only raise)")
    print("   Exit  : Trailing stop hit  OR  Z > 0")
    print("#"*80)


# ====================== DAILY COMMAND ======================
if __name__ == "__main__":
    live_status_all()


################################################################################
DUAL-MODE LIVE DASHBOARD   |   2026-07-12 23:14
################################################################################

████████████████████████████████████████████████████████████████████████████████
  MOMENTUM BUCKET  →  Original Strategy (No Filter)
  NVDA, META, NET
████████████████████████████████████████████████████████████████████████████████

  NVDA   |   MOMENTUM (No Filter)
  Current Price      : $210.96
  Z-score            : 1.48   (entry when < -1.5)
  20-SMA             : $201.95
  50-SMA             : $209.07   (Trend OK: True)
  ATR(14)            : $6.80
------------------------------------------------------------------------
  BUY TRIGGER        : $192.83
     → This is where Z-score = -1.5

  IF YOU BUY AT THE TRIGGER:
     Entry Price         : $192.83
     Initial Stop Loss   : $179.23
     Mean-Reversion Exit : $201.95  (Z > 0)
     Risk / Reward       : $13.60 / $9.12  →  

In [2]:
live_status_all()


################################################################################
DUAL-MODE LIVE DASHBOARD   |   2026-07-12 23:14
################################################################################

████████████████████████████████████████████████████████████████████████████████
  MOMENTUM BUCKET  →  Original Strategy (No Filter)
  NVDA, META, NET
████████████████████████████████████████████████████████████████████████████████

  NVDA   |   MOMENTUM (No Filter)
  Current Price      : $210.96
  Z-score            : 1.48   (entry when < -1.5)
  20-SMA             : $201.95
  50-SMA             : $209.07   (Trend OK: True)
  ATR(14)            : $6.80
------------------------------------------------------------------------
  BUY TRIGGER        : $192.83
     → This is where Z-score = -1.5

  IF YOU BUY AT THE TRIGGER:
     Entry Price         : $192.83
     Initial Stop Loss   : $179.23
     Mean-Reversion Exit : $201.95  (Z > 0)
     Risk / Reward       : $13.60 / $9.12  →  

In [3]:
# ============================================================
# DUAL-MODE LIVE DASHBOARD - Compact Version
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ====================== SETTINGS ======================
Z_ENTRY     = -1.5
ATR_MULT    = 2.0
SMA_WINDOW  = 20
ATR_WINDOW  = 14
TREND_SMA   = 50

MOMENTUM_BUCKET = ["NVDA", "META", "NET"]
QUALITY_BUCKET  = ["NOW", "MSFT", "GOOGL", "PANW", "CRWD", "DDOG", "CRM"]
# ======================================================


def get_data(ticker, days=120):
    end = datetime.now().date() + timedelta(days=1)
    start = end - timedelta(days=days)
    try:
        df = yf.download(ticker, start=start.strftime('%Y-%m-%d'),
                         end=end.strftime('%Y-%m-%d'),
                         multi_level_index=False, progress=False, auto_adjust=True)
        if df.empty:
            return None
        return df[['Open', 'High', 'Low', 'Close']].dropna().copy()
    except:
        return None


def get_levels(ticker, use_filter):
    df = get_data(ticker)
    if df is None or len(df) < 60:
        return None

    df = df.copy()
    df['sma'] = df['Close'].rolling(SMA_WINDOW).mean()
    df['std'] = df['Close'].rolling(SMA_WINDOW).std()
    df['zscore'] = (df['Close'] - df['sma']) / df['std']
    df['trend_sma'] = df['Close'].rolling(TREND_SMA).mean()

    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - df['Close'].shift(1)).abs(),
        (df['Low'] - df['Close'].shift(1)).abs()
    ], axis=1).max(axis=1)
    df['atr'] = tr.rolling(ATR_WINDOW).mean()

    last = df.iloc[-1]
    close = float(last['Close'])
    sma = float(last['sma'])
    std = float(last['std'])
    atr = float(last['atr'])
    z = float(last['zscore'])
    trend_sma = float(last['trend_sma']) if pd.notna(last['trend_sma']) else np.nan

    buy_trigger = sma + (Z_ENTRY * std)
    initial_stop = buy_trigger - (ATR_MULT * atr)
    mean_exit = sma

    trend_ok = True
    if use_filter and not np.isnan(trend_sma):
        trend_ok = close > trend_sma

    signal = (z < Z_ENTRY) and trend_ok
    dist_pct = (close - buy_trigger) / close * 100

    return {
        'ticker': ticker,
        'close': close,
        'z': z,
        'buy_trigger': buy_trigger,
        'initial_stop': initial_stop,
        'mean_exit': mean_exit,
        'trend_ok': trend_ok,
        'use_filter': use_filter,
        'signal': signal,
        'dist_pct': dist_pct,
        'risk': buy_trigger - initial_stop,
        'reward': mean_exit - buy_trigger
    }


def live_dashboard():
    print("\n" + "="*95)
    print(f"DUAL-MODE LIVE DASHBOARD                    {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print("="*95)

    rows = []

    for t in MOMENTUM_BUCKET:
        info = get_levels(t, use_filter=False)
        if info:
            info['bucket'] = "Momentum"
            rows.append(info)

    for t in QUALITY_BUCKET:
        info = get_levels(t, use_filter=True)
        if info:
            info['bucket'] = "Quality"
            rows.append(info)

    # ----- SUMMARY TABLE -----
    print(f"\n{'Ticker':<7} {'Bucket':<10} {'Price':>9} {'Z':>7} {'Trigger':>9} {'Stop':>9} {'Exit':>9} {'Dist%':>7} {'Signal'}")
    print("-"*95)

    for r in rows:
        sig = "← BUY" if r['signal'] else ""
        print(f"{r['ticker']:<7} {r['bucket']:<10} ${r['close']:>8.2f} {r['z']:>7.2f} "
              f"${r['buy_trigger']:>8.2f} ${r['initial_stop']:>8.2f} ${r['mean_exit']:>8.2f} "
              f"{r['dist_pct']:>6.1f}% {sig}")

    print("-"*95)

    # ----- DETAILS only for close or active signals -----
    interesting = [r for r in rows if r['signal'] or r['dist_pct'] < 12]

    if interesting:
        print("\nDETAILED LEVELS (active signal or within 12% of trigger)")
        print("="*95)
        for r in interesting:
            mode = "Quality + 50SMA filter" if r['use_filter'] else "Momentum (no filter)"
            print(f"\n{r['ticker']}   [{mode}]")
            print(f"  Price {r['close']:.2f}  |  Z {r['z']:.2f}")
            print(f"  BUY TRIGGER   : ${r['buy_trigger']:.2f}")
            print(f"  Initial Stop  : ${r['initial_stop']:.2f}")
            print(f"  Mean-Rev Exit : ${r['mean_exit']:.2f}  (Z>0)")
            print(f"  Risk/Reward   : ${r['risk']:.2f} / ${r['reward']:.2f}")
            if r['signal']:
                print(f"  >>> ALL-IN BUY SIGNAL <<<")
    else:
        print("\nNo stocks are close to a buy trigger right now (all > 12% away).")

    print("\n" + "="*95)
    print("RULES:  Momentum = Z<-1.5 only  |  Quality = Z<-1.5 AND Close>50SMA")
    print("        Both: Trail stop = Close-2×ATR  |  Also exit when Z>0")
    print("="*95)


# Run daily
live_dashboard()


DUAL-MODE LIVE DASHBOARD                    2026-07-12 23:15

Ticker  Bucket         Price       Z   Trigger      Stop      Exit   Dist% Signal
-----------------------------------------------------------------------------------------------
NVDA    Momentum   $  210.96    1.48 $  192.83 $  179.23 $  201.95    8.6% 
META    Momentum   $  669.21    2.71 $  537.75 $  484.07 $  584.55   19.6% 
NET     Momentum   $  268.40    1.54 $  213.84 $  187.99 $  240.80   20.3% 
NOW     Quality    $  107.71    1.05 $   92.14 $   80.85 $  101.31   14.5% 
MSFT    Quality    $  385.10    0.40 $  363.36 $  338.00 $  380.51    5.6% 
GOOGL   Quality    $  357.18   -0.07 $  343.69 $  320.72 $  357.88    3.8% 
PANW    Quality    $  325.91    0.55 $  267.66 $  233.15 $  310.29   17.9% 
CRWD    Quality    $  187.18    0.55 $  163.40 $  144.00 $  180.76   12.7% 
DDOG    Quality    $  257.54    0.91 $  215.94 $  190.24 $  241.84   16.2% 
CRM     Quality    $  163.32    0.52 $  150.79 $  138.52 $  160.09    7.7% 